In [3]:
import pandas as pd

df = pd.read_csv(
    "https://svn.vsp.tu-berlin.de/repos/public-svn/matsim/scenarios/countries/de/berlin/berlin-v6.4/output/berlin-v6.4-10pct/berlin-v6.4.output_trips.csv.gz",
    sep=";", low_memory=False)


In [4]:
df.columns

Index(['person', 'trip_number', 'trip_id', 'dep_time', 'trav_time',
       'wait_time', 'traveled_distance', 'euclidean_distance', 'main_mode',
       'longest_distance_mode', 'modes', 'start_activity_type',
       'end_activity_type', 'start_facility_id', 'start_link', 'start_x',
       'start_y', 'end_facility_id', 'end_link', 'end_x', 'end_y',
       'first_pt_boarding_stop', 'last_pt_egress_stop'],
      dtype='str')

Let's find out, what kind of trip chains we have.

In [5]:
trip_chains = (
    df.groupby("person")["main_mode"]
    .agg(lambda s: "_".join(s.astype(str)))
    .reset_index(name="mode_chain")
)

In [6]:
# number of trips per chain
trip_chains["n_trips"] = trip_chains["mode_chain"].str.count("_") + 1

# frequency of each chain pattern
chain_counts = (
    trip_chains["mode_chain"]
    .value_counts()
    .reset_index()
)

In [16]:
def mode_before_pt(chain):
    modes = chain.split("_")

    try:
        pt_idx = modes.index("pt")  # first pt occurrence
        if pt_idx > 0:
            return modes[pt_idx - 1]
    except ValueError:
        pass

    return None


trip_chains["before_pt"] = trip_chains["mode_chain"].apply(mode_before_pt)

counts = (
    trip_chains["before_pt"]
    .isin(["car", "ride", "bike"])
    .sum()
)

print(counts)

3319


In [17]:
counts / chain_counts["count"].sum()

np.float64(0.0069905767278456215)